# Spending habits

Window: August 2025 through September 2026, excluding July and August 2026. That is 12 months with spend. October 2026 is two days of export and stays out.

Spend is rows with `include_in_spend` true, same rule as `purchases.ipynb`. Refunds and payments are out of the totals and called out where they matter. Monthly averages divide by 12. The worker briefs said 11; the swarm proved 12, and every average below uses 12.

Built by a 4-worker swarm: trends, food, discretionary, recurring. Each section is that worker's cells, unchanged except divisor and display fixes.

In [ ]:
import io
import sys
from pathlib import Path

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from IPython.display import Image, display

def show(fig):
    buf = io.BytesIO()
    fig.savefig(buf, format="png", bbox_inches="tight")
    plt.close(fig)
    display(Image(data=buf.getvalue()))

ROOT = Path.cwd()
if not (ROOT / "src" / "analyze_purchases.py").exists():
    ROOT = Path("/Users/alex/Code/Purchases")
sys.path.insert(0, str(ROOT / "src"))

import pandas as pd
from analyze_purchases import sum_amount

DATA_DIR = Path("/Users/alex/Code/Purchases")
df = pd.read_csv(DATA_DIR / "data" / "cleaned_transactions.csv")
if df["include_in_spend"].dtype != bool:
    df["include_in_spend"] = df["include_in_spend"].astype(str).str.lower().isin(["true", "1"])
in_window = (
    (df["txn_date"] >= "2025-08-01") & (df["txn_date"] <= "2026-09-30")
    & ~df["txn_date"].between("2026-07-01", "2026-08-31")
)
past = df[in_window & df["include_in_spend"]].copy()
uni = pd.read_csv(DATA_DIR / "data" / "unified_transactions.csv")
if uni["include_in_spend"].dtype != bool:
    uni["include_in_spend"] = uni["include_in_spend"].astype(str).str.lower().isin(["true", "1"])
uin_window = (
    (uni["txn_date"] >= "2025-08-01") & (uni["txn_date"] <= "2026-09-30")
    & ~uni["txn_date"].between("2026-07-01", "2026-08-31")
)
upast = uni[uin_window].copy()
print(f"window spend {sum_amount(past['amount']):.2f} on {len(past)} rows")
print(f"months with spend: {len(sorted(past['txn_date'].str.slice(0, 7).unique()))}")


## Monthly spend and run-rate

In [ ]:
import pandas as pd

frame = past.copy()
frame["month"] = pd.to_datetime(frame["txn_date"]).dt.strftime("%Y-%m")
months = sorted(frame["month"].unique())
rows = []
for month in months:
    sub = frame[frame["month"] == month]
    rows.append({
        "month": month,
        "spend": sum_amount(sub["amount"]),
        "purchases": int(len(sub)),
    })
monthly = pd.DataFrame(rows)
total = sum_amount(frame["amount"])
run_rate = total / len(monthly)
fig, ax = plt.subplots(figsize=(10, 4))
ax.bar(monthly["month"], monthly["spend"], color="#4c78a8")
ax.axhline(run_rate, color="#e45756", linestyle="--", label=f"average ${run_rate:,.2f}")
ax.set_title("Monthly spend")
ax.set_ylabel("dollars")
ax.tick_params(axis="x", rotation=45)
ax.legend()
fig.tight_layout()
show(fig)
print(f"window ${total:,.2f} over {len(monthly)} months; run-rate ${run_rate:,.2f}")
print(
    f"highest {monthly.loc[monthly['spend'].idxmax(), 'month']} "
    f"${monthly['spend'].max():,.2f}; "
    f"lowest {monthly.loc[monthly['spend'].idxmin(), 'month']} "
    f"${monthly['spend'].min():,.2f}"
)
display(monthly)


## Category share of window spend

In [ ]:
import pandas as pd

total = sum_amount(past["amount"])
rows = []
for category, sub in past.groupby("category"):
    spend = sum_amount(sub["amount"])
    rows.append({
        "category": category,
        "spend": spend,
        "share_pct": round(spend / total * 100, 2),
    })
share = pd.DataFrame(rows).sort_values("spend", ascending=False).reset_index(drop=True)
display(share)
fig, ax = plt.subplots(figsize=(8, 4.5))
plot = share.iloc[::-1]
ax.barh(plot["category"], plot["spend"], color="#4c78a8")
ax.set_xlabel("dollars")
ax.set_title("Category share of window spend")
fig.tight_layout()
show(fig)
dining = share.loc[share["category"] == "dining"].iloc[0]
print(f"dining ${dining['spend']:,.2f}, {dining['share_pct']:.2f} percent of ${total:,.2f}")


## Top 10 dining merchants by spend

In [ ]:
import pandas as pd

flag = upast["include_in_spend"]
if flag.dtype == object:
    flag = flag.astype(str).str.lower().eq("true")
dining = upast.loc[flag & (upast["category"] == "dining")].copy()
dining = dining.sort_values(["txn_date", "description"])

top = (
    dining.groupby("merchant_key", sort=False)
    .agg(
        merchant=("merchant_key", "first"),
        purchases=("amount", "size"),
        spend=("amount", sum_amount),
        example=("description", "first"),
    )
    .sort_values(["spend", "purchases"], ascending=[False, False])
    .head(10)
)
top["avg_ticket"] = (top["spend"] / top["purchases"]).round(2)
top = top[["merchant", "purchases", "spend", "avg_ticket", "example"]].reset_index(drop=True)
display(top)

plot = top.iloc[::-1]
fig, ax = plt.subplots(figsize=(9, 6))
ax.barh(plot["merchant"], plot["spend"], color="#4c78a8")
ax.set_title("Top 10 dining merchants by spend")
ax.set_xlabel("Spend")
fig.tight_layout()
show(fig)


## Dining on weekdays and weekends

In [ ]:
import pandas as pd

flag = upast["include_in_spend"]
if flag.dtype == object:
    flag = flag.astype(str).str.lower().eq("true")
dining = upast.loc[flag & (upast["category"] == "dining")].copy()
dow = pd.to_datetime(dining["txn_date"]).dt.dayofweek
dining["daytype"] = dow.map(lambda d: "weekend" if d >= 5 else "weekday")

by_day = pd.DataFrame({
    "spend": dining.groupby("daytype")["amount"].apply(sum_amount),
    "purchases": dining.groupby("daytype").size(),
}).reindex(["weekday", "weekend"])
by_day["monthly_avg"] = (by_day["spend"] / 12).round(2)
by_day = by_day.reset_index()
display(by_day)

fig, ax = plt.subplots(figsize=(6, 4))
ax.bar(by_day["daytype"], by_day["spend"], color="#4c78a8")
ax.set_title("Dining on weekdays and weekends")
ax.set_ylabel("Spend")
ax.set_xlabel("")
fig.tight_layout()
show(fig)


## Biggest discretionary purchases

Largest 15 purchases in shopping, other, travel, and services.

In [ ]:
import pandas as pd

cats = ["shopping", "other", "travel", "services"]
disc = past[past["category"].isin(cats)].copy()
biggest = (
    disc.sort_values(["amount", "txn_date", "description"], ascending=[False, True, True])
    .head(15)[["txn_date", "description", "category", "amount"]]
    .reset_index(drop=True)
)
display(biggest)

labels = biggest["txn_date"] + "  " + biggest["description"].str.slice(0, 36)
fig, ax = plt.subplots(figsize=(8, 6))
ax.barh(labels.iloc[::-1], biggest["amount"].iloc[::-1], color="#4c78a8")
ax.set_xlabel("Amount")
ax.set_title("Biggest 15 discretionary purchases")
fig.tight_layout()
show(fig)


## Repeat merchants

Merchants with at least two purchases, top 10 by spend.

In [ ]:
import pandas as pd

cats = ["shopping", "other", "travel", "services"]
flag = upast["include_in_spend"]
if flag.dtype == object:
    flag = flag.astype(str).str.lower().isin(["true", "1"])
disc = upast[flag & upast["category"].isin(cats)].copy()
repeat = (
    disc.groupby("merchant_key", sort=False)
    .agg(
        merchant=("merchant_display", "first"),
        purchases=("amount", "size"),
        spend=("amount", sum_amount),
    )
    .reset_index(drop=True)
)
repeat = (
    repeat[repeat["purchases"] >= 2]
    .sort_values(["spend", "merchant"], ascending=[False, True])
    .head(10)[["merchant", "purchases", "spend"]]
    .reset_index(drop=True)
)
display(repeat)

plot = repeat.iloc[::-1]
fig, ax = plt.subplots(figsize=(8, 5))
ax.barh(plot["merchant"], plot["spend"], color="#4c78a8")
ax.set_xlabel("Spend")
ax.set_title("Top 10 repeat merchants by spend")
fig.tight_layout()
show(fig)


## Recurring charges

Group window spend by cleaned description. Keep a description that appears in at least 3 distinct months if the amount repeats exactly, or if max minus min is within 15 percent of the mean. A monthly bill has one charge per active month and a median gap of 25 to 35 days, and its monthly load is the price. Every other candidate uses its window total divided by the months in this frame.

In [ ]:
import pandas as pd
from decimal import Decimal, ROUND_HALF_UP

def cents(value):
    if not isinstance(value, Decimal):
        value = Decimal(str(value))
    return float(value.quantize(Decimal("0.01"), rounding=ROUND_HALF_UP))

spend = past.copy()
spend["txn_date"] = spend["txn_date"].astype(str).str.slice(0, 10)
spend["amount"] = pd.to_numeric(spend["amount"])
if "include_in_spend" in spend.columns:
    flag = spend["include_in_spend"]
    if flag.dtype != bool:
        flag = flag.astype(str).str.lower().isin(["true", "1"])
    spend = spend.loc[flag].copy()
keep = (spend["txn_date"] >= "2025-08-01") & (spend["txn_date"] <= "2026-09-30")
trip = (spend["txn_date"] >= "2026-07-01") & (spend["txn_date"] <= "2026-08-31")
spend = spend.loc[keep & ~trip].copy()
spend["month"] = spend["txn_date"].str.slice(0, 7)
window_months = int(spend["month"].nunique())

rows = []
for description, group in spend.groupby("description", sort=False):
    months_active = int(group["month"].nunique())
    if months_active < 3:
        continue
    amounts = [Decimal(f"{value:.2f}") for value in group["amount"]]
    mean = sum(amounts) / len(amounts)
    spread = max(amounts) - min(amounts)
    exact = len(set(amounts)) == 1
    if not (exact or spread <= mean * Decimal("0.15")):
        continue
    total = sum_amount(group["amount"])
    per_charge = float(amounts[0]) if exact else cents(mean)
    dates = pd.to_datetime(sorted(group["txn_date"]))
    median_gap = None
    if len(dates) >= 2:
        median_gap = float(dates.to_series().diff().dt.days.dropna().median())
    monthly_bill = (
        len(group) == months_active
        and median_gap is not None
        and 25 <= median_gap <= 35
    )
    if monthly_bill:
        monthly_load = per_charge
        cadence = "monthly"
    else:
        monthly_load = cents(Decimal(f"{total:.2f}") / Decimal(window_months))
        cadence = "window_average"
    rows.append(
        {
            "description": description,
            "months_active": months_active,
            "charges": int(len(group)),
            "per_charge": per_charge,
            "monthly_load": monthly_load,
            "cadence": cadence,
        }
    )

recurring = pd.DataFrame(rows)
if recurring.empty:
    recurring = pd.DataFrame(
        columns=["description", "months_active", "charges", "per_charge", "monthly_load", "cadence"]
    )
else:
    recurring = recurring.sort_values(
        ["monthly_load", "description"], ascending=[False, True]
    ).reset_index(drop=True)
display(recurring)
load_total = Decimal("0")
for value in recurring["monthly_load"]:
    load_total += Decimal(f"{value:.2f}")
print("recurring monthly load", cents(load_total))
print("window months", window_months)

inst = spend[spend["description"].str.startswith("MONTHLY INSTALLMENTS")]
inst_total = sum_amount(inst["amount"]) if len(inst) else 0.0
print("installment rows missed by description grouping", int(len(inst)), inst_total)

fig, ax = plt.subplots(figsize=(8, 3.2))
ax.barh(recurring["description"], recurring["monthly_load"])
ax.set_xlabel("Monthly load, dollars")
ax.set_title("Monthly load by recurring charge")
fig.tight_layout()
show(fig)


## Essentials and flexible spend

Essentials are gas, groceries, medical, and transportation. Flexible is every other spend category. The two totals must equal window spend.

In [ ]:
import pandas as pd
from decimal import Decimal, ROUND_HALF_UP

def cents(value):
    if not isinstance(value, Decimal):
        value = Decimal(str(value))
    return float(value.quantize(Decimal("0.01"), rounding=ROUND_HALF_UP))

spend = past.copy()
spend["txn_date"] = spend["txn_date"].astype(str).str.slice(0, 10)
spend["amount"] = pd.to_numeric(spend["amount"])
if "include_in_spend" in spend.columns:
    flag = spend["include_in_spend"]
    if flag.dtype != bool:
        flag = flag.astype(str).str.lower().isin(["true", "1"])
    spend = spend.loc[flag].copy()
keep = (spend["txn_date"] >= "2025-08-01") & (spend["txn_date"] <= "2026-09-30")
trip = (spend["txn_date"] >= "2026-07-01") & (spend["txn_date"] <= "2026-08-31")
spend = spend.loc[keep & ~trip].copy()
spend["month"] = spend["txn_date"].str.slice(0, 7)
window_months = int(spend["month"].nunique())

ess_cats = ["gas", "groceries", "medical", "transportation"]
ess_amount = spend.loc[spend["category"].isin(ess_cats), "amount"]
flex_amount = spend.loc[~spend["category"].isin(ess_cats), "amount"]
ess_total = sum_amount(ess_amount)
flex_total = sum_amount(flex_amount)
direct = sum_amount(spend["amount"])
combined = sum_amount(pd.concat([ess_amount, flex_amount]))
if combined != direct:
    raise RuntimeError(f"essentials plus flexible {combined} != direct {direct}")

def share(part, whole):
    return float(
        (Decimal(f"{part:.2f}") / Decimal(f"{whole:.2f}")).quantize(
            Decimal("0.0001"), rounding=ROUND_HALF_UP
        )
    )

rows = []
for bucket, total in (("essentials", ess_total), ("flexible", flex_total), ("total", direct)):
    rows.append(
        {
            "bucket": bucket,
            "total": total,
            "monthly_average": cents(Decimal(f"{total:.2f}") / Decimal(window_months)),
            "share_of_spend": 1.0 if bucket == "total" else share(total, direct),
        }
    )
split = pd.DataFrame(rows)
display(split)

cat_rows = []
for category, group in spend.groupby("category"):
    total = sum_amount(group["amount"])
    cat_rows.append(
        {
            "category": category,
            "bucket": "essentials" if category in ess_cats else "flexible",
            "total": total,
            "share_of_spend": share(total, direct),
        }
    )
by_cat = pd.DataFrame(cat_rows).sort_values("total", ascending=False).reset_index(drop=True)
display(by_cat)

fig, ax = plt.subplots()
ax.pie(
    [ess_total, flex_total],
    labels=["essentials", "flexible"],
    autopct="%1.1f%%",
)
ax.set_title("Essentials and flexible share of window spend")
fig.tight_layout()
show(fig)
print("direct", direct, "essentials", ess_total, "flexible", flex_total, "months", window_months)


## What to cut

Repeatable habits first, biggest monthly savings first. Worker /11 figures are scaled to /12 months with spend.

| Cut | Saves/mo | Why |
| --- | --- | --- |
| Fast-food chains (In-N-Out, Taco Bell, Raising Canes, McDonald's, Chick-fil-A) | 543.02 | 54 repeat purchases, not one-off dinners. |
| Uniqlo | 406.13 | 8 apparel purchases across the window. |
| Other tea and boba (3cat, Sunright, Omomo + 14 more) | 389.45 | Excludes Heytea below. |
| Skydive Elsinore | 333.67 | One outing, two charges. Not a habit; cap one-offs. |
| K-Top Hair Salon | 270.51 | 6 visits at $48-53. All K-Top descriptions, not just the stable one. |
| EM 7 Grams | 262.21 | 6 sit-down visits, $47.68 ticket. |
| Heytea | 259.61 | 26 purchases. Starbucks is only 5 for $56.40. |
| Boiling Point | 254.01 | 7 visits, $39.59 ticket. |
| Spotify | 16.99 | Flat $16.99/mo subscription. 11 charges in window. |
| Regal | 8.00 | About $8.00/mo on this description. Skip-a-movie cut. |
| **Habit total** | **{habit_total:.2f}** | **Excludes overlaps below** |

Different framing, same dollars, do not add: bringing weekend dining down to the weekday daily rate saves about $140.09/mo, and halving single-visit dining saves about $140.01/mo. Both overlap the merchant rows above.

One-offs to cap, not monthly bills: September one-offs (Orient watch, Dr. Martens, Elorea) $681.67 in one month; the $399 Frontier charge posted 2026-09-30 and was refunded 2026-10-02, outside the window; Avis was one 2025 cluster ($561.20), not ongoing. The $149.91 Apple installment posts through 12 of 12, then ends.

Not the problem: subscriptions are $37.16/mo against $1,201.35/mo flexible. Essentials are 19.4% of spend. Groceries are $71.75/mo because eating out ($567.02/mo dining) replaces grocery trips, not because groceries are cheap.

## Saturdays

Same window and exclusions. Saturday is the priciest day of the week: dining plus the two biggest one-offs in the file both land on Saturdays.

In [ ]:
import pandas as pd

saturday = past[pd.to_datetime(past["txn_date"]).dt.dayofweek == 5].copy()
usaturday = upast[pd.to_datetime(upast["txn_date"]).dt.dayofweek == 5].copy()
all_days = pd.date_range("2025-08-01", "2026-09-30")
all_days = all_days[~((all_days >= "2026-07-01") & (all_days <= "2026-08-31"))]
n_saturdays = int((all_days.dayofweek == 5).sum())
sat_total = sum_amount(saturday["amount"])
print(f"saturday spend {sat_total:.2f} on {len(saturday)} purchases")
print(f"average {sat_total / n_saturdays:.2f} per Saturday over {n_saturdays} Saturdays")
print(f"average {sat_total / saturday['txn_date'].nunique():.2f} per Saturday with purchases")

by_sat_cat = (
    saturday.groupby("category")["amount"]
    .apply(sum_amount)
    .sort_values(ascending=False)
    .rename("spend")
)
display(by_sat_cat.to_frame())

fig, ax = plt.subplots(figsize=(8, 4))
by_sat_cat.sort_values().plot.barh(ax=ax, color="#4c78a8", legend=False)
ax.set_title("Saturday spend by category")
ax.set_xlabel("Spend")
ax.set_ylabel("")
fig.tight_layout()
show(fig)


## Saturday transaction counts

How many times money moves on Saturdays, per category. Averages divide by all 52 Saturdays in the window, quiet ones included.

In [ ]:
import pandas as pd

saturday = past[pd.to_datetime(past["txn_date"]).dt.dayofweek == 5].copy()
all_days = pd.date_range("2025-08-01", "2026-09-30")
all_days = all_days[~((all_days >= "2026-07-01") & (all_days <= "2026-08-31"))]
n_saturdays = int((all_days.dayofweek == 5).sum())

counts = (
    saturday.groupby("category").size()
    .sort_values(ascending=False)
    .rename("transactions")
    .to_frame()
)
counts["per_saturday"] = (counts["transactions"] / n_saturdays).round(2)
display(counts)
print(f"{len(saturday)} Saturday transactions, {len(saturday) / n_saturdays:.2f} per Saturday")

fig, ax = plt.subplots(figsize=(8, 4))
counts["transactions"].sort_values().plot.barh(ax=ax, color="#4c78a8", legend=False)
ax.set_title("Saturday transactions by category")
ax.set_xlabel("Transactions")
ax.set_ylabel("")
fig.tight_layout()
show(fig)


# Re-run

From the project root:

```bash
.venv/bin/python -m jupyter notebook notebooks/spending_habits.ipynb
```

Worker evidence lives in `swarm-out/cut-analysis/w1_trends.json` through `w4_recurring.json` at the project root. The notebook reads `data/cleaned_transactions.csv` and `data/unified_transactions.csv` and filters the window in the setup cell.